# 1. Analysis - Income Statement

## Notebook Summary

This notebook analyzes a single company’s income statement history from Financial Modeling Prep using the shared ticker in `../../params.py`. It is organized into data loading and preparation, summary snapshots, core trend charts, seasonal growth views, margin-driver diagnostics, and revenue segmentation where FMP data is available.

Run the notebook from top to bottom after updating `ticker_str` in `../../params.py` so the helper functions, annual and quarterly datasets, and all downstream charts stay in sync.

## Data Loading and Preparation

These cells initialize the FMP helpers, normalize the ticker, load annual and quarterly income statement history, and calculate the derived growth, margin, and ratio fields used throughout the notebook.

They also build the base annual and quarterly summary tables that the rest of the analysis depends on.

In [ ]:
# 2. Setup and FMP helpers
from pathlib import Path
import os
import runpy

import pandas as pd
import plotly.graph_objects as go
import requests


PARAMS_FILE_CANDIDATES = []
for base_path in [Path.cwd(), *Path.cwd().parents]:
    PARAMS_FILE_CANDIDATES.extend(
        [
            base_path / "params.py",
            base_path / "Single Asset Profile" / "params.py",
            base_path / "Notebooks" / "Single Asset Profile" / "params.py",
            base_path / "Investment Scripts" / "Notebooks" / "Single Asset Profile" / "params.py",
        ]
    )

for params_file in PARAMS_FILE_CANDIDATES:
    if params_file.exists():
        break
else:
    raise FileNotFoundError("Could not locate params.py in the Single Asset Profile workspace.")

notebook_params = runpy.run_path(str(params_file))
single_asset_params = notebook_params["get_single_asset_profile_params"]()

TICKER = single_asset_params["ticker_str"]
BASE_URL = "https://financialmodelingprep.com/stable"
ANNUAL_LIMIT = 20
QUARTERLY_LIMIT = 40


def find_project_root(start_path: Path | None = None) -> Path:
    current = (start_path or Path.cwd()).resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
            return candidate
    return current


def load_project_env() -> None:
    env_path = find_project_root() / ".env"
    if not env_path.exists():
        return

    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if value and value[0] == value[-1] and value[0] in {'"', "'"}:
            value = value[1:-1]
        os.environ.setdefault(key, value)


def get_api_key() -> str:
    load_project_env()
    api_key = os.getenv("FMP_API_KEY") or os.getenv("FINANCIAL_MODELING_PREP_API_KEY")
    if api_key:
        return api_key
    raise KeyError("Missing FMP_API_KEY in the project .env file or environment.")


def normalize_symbol(ticker: str) -> str:
    normalized = str(ticker).strip().upper()
    for old, new in ((chr(92), "."), ("/", "."), ("-", ".")):
        normalized = normalized.replace(old, new)
    return normalized


def request_json(endpoint: str, **params):
    response = requests.get(
        f"{BASE_URL}/{endpoint}",
        params={**params, "apikey": get_api_key()},
        timeout=30,
    )
    response.raise_for_status()
    payload = response.json()
    if isinstance(payload, dict) and payload.get("Error Message"):
        raise RuntimeError(payload["Error Message"])
    return payload


def prepare_income_statement_frame(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if frame.empty:
        return frame

    frame["date"] = pd.to_datetime(frame.get("date"), errors="coerce")

    if "calendarYear" in frame.columns:
        frame["calendarYear"] = pd.to_numeric(frame["calendarYear"], errors="coerce")
    else:
        frame["calendarYear"] = pd.NA

    if frame["calendarYear"].isna().all():
        frame["calendarYear"] = frame["date"].dt.year

    if "period" not in frame.columns:
        frame["period"] = pd.NA

    numeric_columns = [
        "revenue",
        "grossProfit",
        "operatingIncome",
        "incomeBeforeTax",
        "incomeTaxExpense",
        "totalOtherIncomeExpensesNet",
        "netIncome",
        "costOfRevenue",
        "operatingExpenses",
        "generalAndAdministrativeExpenses",
        "sellingAndMarketingExpenses",
        "sellingGeneralAndAdministrativeExpenses",
        "researchAndDevelopmentExpenses",
        "eps",
        "epsDiluted",
        "weightedAverageShsOut",
        "weightedAverageShsOutDil",
    ]
    for column in numeric_columns:
        if column not in frame.columns:
            frame[column] = pd.NA
        frame[column] = pd.to_numeric(frame[column], errors="coerce")

    return frame


def apply_standard_figure_layout(fig: go.Figure, title: str, height: int, bottom_margin: int = 140) -> None:
    fig.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_dark",
        paper_bgcolor="#020817",
        plot_bgcolor="#0f172a",
        font={"color": "#e2e8f0"},
        hovermode="x unified",
        hoverlabel={
            "bgcolor": "#0f172a",
            "font": {"color": "#e2e8f0", "size": 13},
            "namelength": -1,
        },
        legend={
            "orientation": "h",
            "yanchor": "top",
            "y": -0.16,
            "x": 0,
            "xanchor": "left",
            "bgcolor": "rgba(2, 8, 23, 0.6)",
        },
        autosize=True,
        height=height,
        margin={"l": 60, "r": 30, "t": 120, "b": bottom_margin},
    )


SYMBOL = normalize_symbol(TICKER)
SYMBOL

In [ ]:
# 3. Load annual income statement history
quote_snapshot = request_json("quote", symbol=SYMBOL)
company_name = quote_snapshot[0].get("name", SYMBOL) if quote_snapshot else SYMBOL
chart_label = f"{company_name} ({SYMBOL})" if company_name != SYMBOL else SYMBOL

annual_income_statement = prepare_income_statement_frame(
    pd.DataFrame(request_json("income-statement", symbol=SYMBOL, limit=ANNUAL_LIMIT))
)

if annual_income_statement.empty:
    raise RuntimeError(f"FMP did not return annual income statement history for {SYMBOL}.")

annual_revenue = (
    annual_income_statement.loc[:, [
        "date",
        "calendarYear",
        "period",
        "revenue",
        "grossProfit",
        "operatingIncome",
        "incomeBeforeTax",
        "incomeTaxExpense",
        "totalOtherIncomeExpensesNet",
        "netIncome",
        "costOfRevenue",
        "operatingExpenses",
        "generalAndAdministrativeExpenses",
        "sellingAndMarketingExpenses",
        "sellingGeneralAndAdministrativeExpenses",
        "researchAndDevelopmentExpenses",
        "eps",
        "epsDiluted",
        "weightedAverageShsOut",
        "weightedAverageShsOutDil",
    ]]
    .dropna(subset=["date", "revenue"])
)
annual_revenue["costOfRevenue"] = annual_revenue["costOfRevenue"].fillna(
    annual_revenue["revenue"] - annual_revenue["grossProfit"]
)
annual_revenue["incomeBeforeTax"] = annual_revenue["incomeBeforeTax"].fillna(
    annual_revenue["netIncome"] + annual_revenue["incomeTaxExpense"]
)
annual_revenue["totalOtherIncomeExpensesNet"] = annual_revenue["totalOtherIncomeExpensesNet"].fillna(
    annual_revenue["incomeBeforeTax"] - annual_revenue["operatingIncome"]
)
annual_revenue["epsDiluted"] = annual_revenue["epsDiluted"].fillna(annual_revenue["eps"])
annual_revenue["weightedAverageShsOutDil"] = annual_revenue["weightedAverageShsOutDil"].fillna(
    annual_revenue["weightedAverageShsOut"]
)
annual_revenue = annual_revenue.sort_values("date").reset_index(drop=True)
annual_revenue["revenueYoY"] = annual_revenue["revenue"].pct_change()
annual_revenue["grossProfitYoY"] = annual_revenue["grossProfit"].pct_change()
annual_revenue["operatingIncomeYoY"] = annual_revenue["operatingIncome"].pct_change()
annual_revenue["incomeBeforeTaxYoY"] = annual_revenue["incomeBeforeTax"].pct_change()
annual_revenue["incomeTaxExpenseYoY"] = annual_revenue["incomeTaxExpense"].pct_change()
annual_revenue["netIncomeYoY"] = annual_revenue["netIncome"].pct_change()
annual_revenue["costOfRevenueYoY"] = annual_revenue["costOfRevenue"].pct_change()
annual_revenue["operatingExpensesYoY"] = annual_revenue["operatingExpenses"].pct_change()
annual_revenue["gaYoY"] = annual_revenue["generalAndAdministrativeExpenses"].pct_change()
annual_revenue["marketingYoY"] = annual_revenue["sellingAndMarketingExpenses"].pct_change()
annual_revenue["sgaYoY"] = annual_revenue["sellingGeneralAndAdministrativeExpenses"].pct_change()
annual_revenue["rdYoY"] = annual_revenue["researchAndDevelopmentExpenses"].pct_change()
annual_revenue["epsYoY"] = annual_revenue["eps"].pct_change()
annual_revenue["epsDilutedYoY"] = annual_revenue["epsDiluted"].pct_change()
annual_revenue["dilutedSharesYoY"] = annual_revenue["weightedAverageShsOutDil"].pct_change()
annual_revenue["grossMargin"] = annual_revenue["grossProfit"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["operatingMargin"] = annual_revenue["operatingIncome"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["incomeBeforeTaxMargin"] = annual_revenue["incomeBeforeTax"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["netMargin"] = annual_revenue["netIncome"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["grossProfitPctRevenue"] = annual_revenue["grossProfit"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["costOfRevenuePctRevenue"] = annual_revenue["costOfRevenue"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["operatingExpensesPctRevenue"] = annual_revenue["operatingExpenses"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["gaPctRevenue"] = annual_revenue["generalAndAdministrativeExpenses"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["marketingPctRevenue"] = annual_revenue["sellingAndMarketingExpenses"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["sgaPctRevenue"] = annual_revenue["sellingGeneralAndAdministrativeExpenses"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["rdPctRevenue"] = annual_revenue["researchAndDevelopmentExpenses"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["otherIncomeExpensePctRevenue"] = annual_revenue["totalOtherIncomeExpensesNet"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["incomeTaxExpensePctRevenue"] = annual_revenue["incomeTaxExpense"].div(annual_revenue["revenue"].replace(0, pd.NA))
annual_revenue["incomeTaxRate"] = annual_revenue["incomeTaxExpense"].div(
    annual_revenue["incomeBeforeTax"].replace(0, pd.NA)
)
annual_revenue["dilutionPct"] = annual_revenue["weightedAverageShsOutDil"].div(
    annual_revenue["weightedAverageShsOut"].replace(0, pd.NA)
) - 1
annual_revenue["grossMarginChange"] = annual_revenue["grossMargin"].diff()
annual_revenue["operatingMarginChange"] = annual_revenue["operatingMargin"].diff()
annual_revenue["netMarginChange"] = annual_revenue["netMargin"].diff()

In [ ]:
# 4. Load quarterly income statement history
quarterly_income_statement = prepare_income_statement_frame(
    pd.DataFrame(request_json("income-statement", symbol=SYMBOL, period="quarter", limit=QUARTERLY_LIMIT))
)

if quarterly_income_statement.empty:
    raise RuntimeError(f"FMP did not return quarterly income statement history for {SYMBOL}.")

quarterly_revenue = (
    quarterly_income_statement.loc[:, [
        "date",
        "calendarYear",
        "period",
        "revenue",
        "grossProfit",
        "operatingIncome",
        "incomeBeforeTax",
        "incomeTaxExpense",
        "totalOtherIncomeExpensesNet",
        "netIncome",
        "costOfRevenue",
        "operatingExpenses",
        "generalAndAdministrativeExpenses",
        "sellingAndMarketingExpenses",
        "sellingGeneralAndAdministrativeExpenses",
        "researchAndDevelopmentExpenses",
        "eps",
        "epsDiluted",
        "weightedAverageShsOut",
        "weightedAverageShsOutDil",
    ]]
    .dropna(subset=["date", "revenue"])
)
quarterly_revenue["costOfRevenue"] = quarterly_revenue["costOfRevenue"].fillna(
    quarterly_revenue["revenue"] - quarterly_revenue["grossProfit"]
)
quarterly_revenue["incomeBeforeTax"] = quarterly_revenue["incomeBeforeTax"].fillna(
    quarterly_revenue["netIncome"] + quarterly_revenue["incomeTaxExpense"]
)
quarterly_revenue["totalOtherIncomeExpensesNet"] = quarterly_revenue["totalOtherIncomeExpensesNet"].fillna(
    quarterly_revenue["incomeBeforeTax"] - quarterly_revenue["operatingIncome"]
)
quarterly_revenue["epsDiluted"] = quarterly_revenue["epsDiluted"].fillna(quarterly_revenue["eps"])
quarterly_revenue["weightedAverageShsOutDil"] = quarterly_revenue["weightedAverageShsOutDil"].fillna(
    quarterly_revenue["weightedAverageShsOut"]
)
quarterly_revenue = quarterly_revenue.sort_values("date").reset_index(drop=True)
quarterly_revenue["quarterLabel"] = quarterly_revenue["period"].astype("string").str.upper().str.extract(r"(Q[1-4])", expand=False)
quarterly_revenue.loc[quarterly_revenue["quarterLabel"].isna(), "quarterLabel"] = (
    "Q" + quarterly_revenue.loc[quarterly_revenue["quarterLabel"].isna(), "date"].dt.quarter.astype("Int64").astype(str)
)
quarterly_revenue["quarterlyYear"] = quarterly_revenue["date"].dt.year
quarterly_revenue["revenueYoY"] = quarterly_revenue["revenue"].pct_change(4)
quarterly_revenue["grossProfitYoY"] = quarterly_revenue["grossProfit"].pct_change(4)
quarterly_revenue["operatingIncomeYoY"] = quarterly_revenue["operatingIncome"].pct_change(4)
quarterly_revenue["incomeBeforeTaxYoY"] = quarterly_revenue["incomeBeforeTax"].pct_change(4)
quarterly_revenue["incomeTaxExpenseYoY"] = quarterly_revenue["incomeTaxExpense"].pct_change(4)
quarterly_revenue["netIncomeYoY"] = quarterly_revenue["netIncome"].pct_change(4)
quarterly_revenue["costOfRevenueYoY"] = quarterly_revenue["costOfRevenue"].pct_change(4)
quarterly_revenue["operatingExpensesYoY"] = quarterly_revenue["operatingExpenses"].pct_change(4)
quarterly_revenue["gaYoY"] = quarterly_revenue["generalAndAdministrativeExpenses"].pct_change(4)
quarterly_revenue["marketingYoY"] = quarterly_revenue["sellingAndMarketingExpenses"].pct_change(4)
quarterly_revenue["sgaYoY"] = quarterly_revenue["sellingGeneralAndAdministrativeExpenses"].pct_change(4)
quarterly_revenue["rdYoY"] = quarterly_revenue["researchAndDevelopmentExpenses"].pct_change(4)
quarterly_revenue["epsYoY"] = quarterly_revenue["eps"].pct_change(4)
quarterly_revenue["epsDilutedYoY"] = quarterly_revenue["epsDiluted"].pct_change(4)
quarterly_revenue["dilutedSharesYoY"] = quarterly_revenue["weightedAverageShsOutDil"].pct_change(4)
quarterly_revenue["ttmRevenue"] = quarterly_revenue["revenue"].rolling(4).sum()
quarterly_revenue["ttmGrossProfit"] = quarterly_revenue["grossProfit"].rolling(4).sum()
quarterly_revenue["ttmOperatingIncome"] = quarterly_revenue["operatingIncome"].rolling(4).sum()
quarterly_revenue["ttmIncomeBeforeTax"] = quarterly_revenue["incomeBeforeTax"].rolling(4).sum()
quarterly_revenue["ttmNetIncome"] = quarterly_revenue["netIncome"].rolling(4).sum()
quarterly_revenue["ttmRevenueYoY"] = quarterly_revenue["ttmRevenue"].pct_change(4)
quarterly_revenue["ttmGrossProfitYoY"] = quarterly_revenue["ttmGrossProfit"].pct_change(4)
quarterly_revenue["ttmOperatingIncomeYoY"] = quarterly_revenue["ttmOperatingIncome"].pct_change(4)
quarterly_revenue["ttmIncomeBeforeTaxYoY"] = quarterly_revenue["ttmIncomeBeforeTax"].pct_change(4)
quarterly_revenue["ttmNetIncomeYoY"] = quarterly_revenue["ttmNetIncome"].pct_change(4)
quarterly_revenue["grossMargin"] = quarterly_revenue["grossProfit"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["operatingMargin"] = quarterly_revenue["operatingIncome"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["incomeBeforeTaxMargin"] = quarterly_revenue["incomeBeforeTax"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["netMargin"] = quarterly_revenue["netIncome"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["grossProfitPctRevenue"] = quarterly_revenue["grossProfit"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["costOfRevenuePctRevenue"] = quarterly_revenue["costOfRevenue"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["operatingExpensesPctRevenue"] = quarterly_revenue["operatingExpenses"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["gaPctRevenue"] = quarterly_revenue["generalAndAdministrativeExpenses"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["marketingPctRevenue"] = quarterly_revenue["sellingAndMarketingExpenses"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["sgaPctRevenue"] = quarterly_revenue["sellingGeneralAndAdministrativeExpenses"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["rdPctRevenue"] = quarterly_revenue["researchAndDevelopmentExpenses"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["otherIncomeExpensePctRevenue"] = quarterly_revenue["totalOtherIncomeExpensesNet"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["incomeTaxExpensePctRevenue"] = quarterly_revenue["incomeTaxExpense"].div(quarterly_revenue["revenue"].replace(0, pd.NA))
quarterly_revenue["incomeTaxRate"] = quarterly_revenue["incomeTaxExpense"].div(
    quarterly_revenue["incomeBeforeTax"].replace(0, pd.NA)
)
quarterly_revenue["dilutionPct"] = quarterly_revenue["weightedAverageShsOutDil"].div(
    quarterly_revenue["weightedAverageShsOut"].replace(0, pd.NA)
) - 1
quarterly_revenue["ttmGrossMargin"] = quarterly_revenue["ttmGrossProfit"].div(
    quarterly_revenue["ttmRevenue"].replace(0, pd.NA)
)
quarterly_revenue["ttmOperatingMargin"] = quarterly_revenue["ttmOperatingIncome"].div(
    quarterly_revenue["ttmRevenue"].replace(0, pd.NA)
)
quarterly_revenue["ttmIncomeBeforeTaxMargin"] = quarterly_revenue["ttmIncomeBeforeTax"].div(
    quarterly_revenue["ttmRevenue"].replace(0, pd.NA)
)
quarterly_revenue["ttmNetMargin"] = quarterly_revenue["ttmNetIncome"].div(
    quarterly_revenue["ttmRevenue"].replace(0, pd.NA)
)
quarterly_revenue["grossMarginChange"] = quarterly_revenue["grossMargin"].diff(4)
quarterly_revenue["operatingMarginChange"] = quarterly_revenue["operatingMargin"].diff(4)
quarterly_revenue["netMarginChange"] = quarterly_revenue["netMargin"].diff(4)

In [ ]:
# 5. Build annual summary stats
latest_annual = annual_revenue.iloc[-1]
annual_years = max(len(annual_revenue) - 1, 1)
annual_cagr = (latest_annual["revenue"] / annual_revenue.iloc[0]["revenue"]) ** (1 / annual_years) - 1 if len(annual_revenue) > 1 else pd.NA

annual_summary = pd.DataFrame(
    [
        {
            "metric": "Latest annual revenue",
            "value": latest_annual["revenue"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual revenue CAGR",
            "value": annual_cagr,
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Latest annual gross margin",
            "value": latest_annual["grossMargin"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Latest annual operating margin",
            "value": latest_annual["operatingMargin"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Latest annual net margin",
            "value": latest_annual["netMargin"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual gross margin change",
            "value": latest_annual["grossMarginChange"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual operating margin change",
            "value": latest_annual["operatingMarginChange"],
            "asOf": latest_annual["date"].date(),
        },
        {
            "metric": "Annual net margin change",
            "value": latest_annual["netMarginChange"],
            "asOf": latest_annual["date"].date(),
        },
    ]
)

In [ ]:
# 6. Build quarterly summary stats
latest_quarter = quarterly_revenue.iloc[-1]
latest_ttm = quarterly_revenue.dropna(subset=["ttmRevenue"]).iloc[-1]

quarterly_summary = pd.DataFrame(
    [
        {
            "metric": "Latest quarterly revenue",
            "value": latest_quarter["revenue"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest TTM revenue",
            "value": latest_ttm["ttmRevenue"],
            "asOf": latest_ttm["date"].date(),
        },
        {
            "metric": "Latest quarterly YoY growth",
            "value": latest_quarter["revenueYoY"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest quarterly gross margin",
            "value": latest_quarter["grossMargin"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest quarterly operating margin",
            "value": latest_quarter["operatingMargin"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Latest quarterly net margin",
            "value": latest_quarter["netMargin"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Quarterly gross margin change",
            "value": latest_quarter["grossMarginChange"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Quarterly operating margin change",
            "value": latest_quarter["operatingMarginChange"],
            "asOf": latest_quarter["date"].date(),
        },
        {
            "metric": "Quarterly net margin change",
            "value": latest_quarter["netMarginChange"],
            "asOf": latest_quarter["date"].date(),
        },
    ]
)

## Trend and Seasonality Charts

These cells move from summary tables into visualization. They cover annual trend charts, quarterly trend charts, and the seasonal quarterly growth view so you can separate long-term operating direction from quarter-specific patterns.

In [ ]:
# 7. Plot annual income statement trends
from plotly.subplots import make_subplots

annual_fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    row_heights=[0.30, 0.22, 0.18, 0.15, 0.15],
    subplot_titles=(
        "Annual Revenue",
        "Annual Gross Profit, Operating Income, and Net Income",
        "Annual Margins",
        "Annual Growth Rates",
        "Annual Margin Changes",
    ),
)

annual_fig.add_trace(
    go.Bar(
        x=annual_revenue["date"],
        y=annual_revenue["revenue"],
        name="Annual revenue",
        marker_color="#60a5fa",
        opacity=0.65,
    ),
    row=1,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["grossProfit"],
        name="Annual gross profit",
        mode="lines+markers",
        line={"color": "#a78bfa", "width": 2.5},
        marker={"size": 7},
    ),
    row=2,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingIncome"],
        name="Annual operating income",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2.5},
        marker={"size": 7},
    ),
    row=2,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["netIncome"],
        name="Annual net income",
        mode="lines+markers",
        line={"color": "#f43f5e", "width": 2.5},
        marker={"size": 7},
    ),
    row=2,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["grossMargin"],
        name="Annual gross margin",
        mode="lines+markers",
        line={"color": "#8b5cf6", "width": 2.5},
        marker={"size": 7},
    ),
    row=3,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingMargin"],
        name="Annual operating margin",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.5},
        marker={"size": 7},
    ),
    row=3,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["netMargin"],
        name="Annual net margin",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2.5},
        marker={"size": 7},
    ),
    row=3,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["revenueYoY"],
        name="Annual revenue growth",
        mode="lines+markers",
        line={"color": "#38bdf8", "width": 2.5},
        marker={"size": 7},
    ),
    row=4,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["grossProfitYoY"],
        name="Annual gross profit growth",
        mode="lines+markers",
        line={"color": "#c084fc", "width": 2.5},
        marker={"size": 7},
    ),
    row=4,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingIncomeYoY"],
        name="Annual operating income growth",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.5},
        marker={"size": 7},
    ),
    row=4,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["netIncomeYoY"],
        name="Annual net income growth",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2.5},
        marker={"size": 7},
    ),
    row=4,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["grossMarginChange"],
        name="Annual gross margin change",
        mode="lines+markers",
        line={"color": "#8b5cf6", "width": 2, "dash": "dash"},
        marker={"size": 6, "symbol": "diamond"},
    ),
    row=5,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingMarginChange"],
        name="Annual operating margin change",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2, "dash": "dash"},
        marker={"size": 6, "symbol": "diamond"},
    ),
    row=5,
    col=1,
)

annual_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["netMarginChange"],
        name="Annual net margin change",
        mode="lines+markers",
        line={"color": "#f43f5e", "width": 2, "dash": "dash"},
        marker={"size": 6, "symbol": "diamond"},
    ),
    row=5,
    col=1,
)

apply_standard_figure_layout(annual_fig, f"{chart_label} annual income statement", 1460)

annual_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=1, col=1)
annual_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=2, col=1)
annual_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=3, col=1)
annual_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=4, col=1)
annual_fig.update_xaxes(title_text="Date", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=5, col=1)

annual_fig.update_yaxes(title_text="Revenue (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
annual_fig.update_yaxes(title_text="Income (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=2, col=1)
annual_fig.update_yaxes(title_text="Margin", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat=".1%", row=3, col=1)
annual_fig.update_yaxes(title_text="Growth", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=4, col=1)
annual_fig.update_yaxes(title_text="Margin change", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=5, col=1)

annual_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 8. Plot quarterly income statement trends
from plotly.subplots import make_subplots

quarterly_fig = make_subplots(
    rows=5,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.045,
    row_heights=[0.30, 0.22, 0.18, 0.15, 0.15],
    subplot_titles=(
        "Quarterly Revenue and TTM Revenue",
        "Quarterly Gross Profit, Operating Income, and Net Income",
        "Quarterly Margins",
        "Quarterly YoY Growth Rates",
        "Quarterly Margin Changes",
    ),
)

quarterly_fig.add_trace(
    go.Bar(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["revenue"],
        name="Quarterly revenue",
        marker_color="#f59e0b",
        opacity=0.45,
    ),
    row=1,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["ttmRevenue"],
        name="TTM revenue",
        mode="lines+markers",
        line={"color": "#22c55e", "width": 3},
        marker={"size": 7},
    ),
    row=1,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["grossProfit"],
        name="Quarterly gross profit",
        mode="lines+markers",
        line={"color": "#a78bfa", "width": 2.5},
        marker={"size": 6},
    ),
    row=2,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingIncome"],
        name="Quarterly operating income",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2.5},
        marker={"size": 6},
    ),
    row=2,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["netIncome"],
        name="Quarterly net income",
        mode="lines+markers",
        line={"color": "#f43f5e", "width": 2.5},
        marker={"size": 6},
    ),
    row=2,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["grossMargin"],
        name="Quarterly gross margin",
        mode="lines+markers",
        line={"color": "#8b5cf6", "width": 2.5},
        marker={"size": 6},
    ),
    row=3,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingMargin"],
        name="Quarterly operating margin",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.5},
        marker={"size": 6},
    ),
    row=3,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["netMargin"],
        name="Quarterly net margin",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2.5},
        marker={"size": 6},
    ),
    row=3,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["revenueYoY"],
        name="Quarterly revenue growth",
        mode="lines+markers",
        line={"color": "#38bdf8", "width": 2.5},
        marker={"size": 6},
    ),
    row=4,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["grossProfitYoY"],
        name="Quarterly gross profit growth",
        mode="lines+markers",
        line={"color": "#c084fc", "width": 2.5},
        marker={"size": 6},
    ),
    row=4,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingIncomeYoY"],
        name="Quarterly operating income growth",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.5},
        marker={"size": 6},
    ),
    row=4,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["netIncomeYoY"],
        name="Quarterly net income growth",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2.5},
        marker={"size": 6},
    ),
    row=4,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["grossMarginChange"],
        name="Quarterly gross margin change",
        mode="lines+markers",
        line={"color": "#8b5cf6", "width": 2, "dash": "dash"},
        marker={"size": 6, "symbol": "diamond"},
    ),
    row=5,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingMarginChange"],
        name="Quarterly operating margin change",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2, "dash": "dash"},
        marker={"size": 6, "symbol": "diamond"},
    ),
    row=5,
    col=1,
)

quarterly_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["netMarginChange"],
        name="Quarterly net margin change",
        mode="lines+markers",
        line={"color": "#f43f5e", "width": 2, "dash": "dash"},
        marker={"size": 6, "symbol": "diamond"},
    ),
    row=5,
    col=1,
)

apply_standard_figure_layout(quarterly_fig, f"{chart_label} quarterly income statement", 1460)

quarterly_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=1, col=1)
quarterly_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=2, col=1)
quarterly_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=3, col=1)
quarterly_fig.update_xaxes(showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=4, col=1)
quarterly_fig.update_xaxes(title_text="Date", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, row=5, col=1)

quarterly_fig.update_yaxes(title_text="Revenue (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=1, col=1)
quarterly_fig.update_yaxes(title_text="Income (USD)", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat="$,.3s", row=2, col=1)
quarterly_fig.update_yaxes(title_text="Margin", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=False, automargin=True, tickformat=".1%", row=3, col=1)
quarterly_fig.update_yaxes(title_text="Growth", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=4, col=1)
quarterly_fig.update_yaxes(title_text="Margin change", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", automargin=True, tickformat=".1%", row=5, col=1)

quarterly_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 9. Plot seasonal quarterly growth rates
from plotly.subplots import make_subplots

seasonal_growth_fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.25, 0.25, 0.25, 0.25],
    subplot_titles=(
        "Revenue YoY Growth by Fiscal Quarter",
        "Gross Profit YoY Growth by Fiscal Quarter",
        "Operating Income YoY Growth by Fiscal Quarter",
        "Net Income YoY Growth by Fiscal Quarter",
    ),
)

quarter_order = ["Q1", "Q2", "Q3", "Q4"]
quarter_colors = {
    "Q1": "#60a5fa",
    "Q2": "#34d399",
    "Q3": "#f59e0b",
    "Q4": "#f87171",
}
metric_rows = [
    ("revenueYoY", "Revenue growth", 1),
    ("grossProfitYoY", "Gross profit growth", 2),
    ("operatingIncomeYoY", "Operating income growth", 3),
    ("netIncomeYoY", "Net income growth", 4),
]

for metric_name, metric_label, row_number in metric_rows:
    for quarter_label in quarter_order:
        seasonal_slice = quarterly_revenue.loc[quarterly_revenue["quarterLabel"] == quarter_label].copy()
        if seasonal_slice.empty:
            continue

        seasonal_growth_fig.add_trace(
            go.Scatter(
                x=seasonal_slice["quarterlyYear"],
                y=seasonal_slice[metric_name],
                mode="lines+markers",
                name=f"{quarter_label} {metric_label}",
                line={"color": quarter_colors[quarter_label], "width": 2.5},
                marker={"size": 7},
                legendgroup=quarter_label,
                showlegend=row_number == 1,
            ),
            row=row_number,
            col=1,
        )

apply_standard_figure_layout(seasonal_growth_fig, f"{chart_label} seasonal quarterly growth rates", 1120)

for row_number in [1, 2, 3, 4]:
    seasonal_growth_fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.18)",
        zeroline=False,
        automargin=True,
        dtick=1,
        row=row_number,
        col=1,
    )
    seasonal_growth_fig.update_yaxes(
        title_text="YoY growth",
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.18)",
        zeroline=True,
        zerolinecolor="rgba(226, 232, 240, 0.35)",
        automargin=True,
        tickformat=".1%",
        row=row_number,
        col=1,
    )

seasonal_growth_fig.update_xaxes(title_text="Fiscal year", row=4, col=1)

seasonal_growth_fig.show(config={"responsive": True, "displaylogo": False})

## Common-Size, EPS, and TTM Profitability

These cells recast the income statement as a share of revenue, isolate per-share performance versus dilution, and track rolling four-quarter profit conversion before moving into the margin-driver charts.

In [ ]:
# 10. Build common-size income statement views
from IPython.display import display

def build_common_size_statement(frame: pd.DataFrame, column_labels: pd.Series) -> pd.DataFrame:
    statement_rows = [
        ("Revenue", pd.Series(1.0, index=frame.index)),
        ("Cost of revenue", -frame["costOfRevenuePctRevenue"]),
        ("Gross profit", frame["grossProfitPctRevenue"]),
        ("Operating expenses", -frame["operatingExpensesPctRevenue"]),
    ]

    optional_rows = [
        ("G&A within opex", -frame["gaPctRevenue"]),
        ("SG&A within opex", -frame["sgaPctRevenue"]),
        ("Marketing within SG&A", -frame["marketingPctRevenue"]),
        ("R&D within opex", -frame["rdPctRevenue"]),
    ]
    for label, values in optional_rows:
        if values.fillna(0).abs().gt(1e-6).any():
            statement_rows.append((label, values))

    statement_rows.extend(
        [
            ("Operating income", frame["operatingMargin"]),
            ("Other income / expense", frame["otherIncomeExpensePctRevenue"]),
            ("Pre-tax income", frame["incomeBeforeTaxMargin"]),
            ("Income tax expense", -frame["incomeTaxExpensePctRevenue"]),
            ("Net income", frame["netMargin"]),
        ]
    )

    statement = pd.DataFrame(
        [values.reset_index(drop=True).tolist() for _, values in statement_rows],
        index=[label for label, _ in statement_rows],
        columns=column_labels.reset_index(drop=True).tolist(),
    )
    return statement.dropna(how="all", axis=0)

annual_common_size_frame = annual_revenue.tail(min(len(annual_revenue), 6)).copy()
annual_common_size_labels = annual_common_size_frame["calendarYear"].astype("Int64").astype(str)
annual_common_size = build_common_size_statement(annual_common_size_frame, annual_common_size_labels)

quarterly_common_size_frame = quarterly_revenue.tail(min(len(quarterly_revenue), 8)).copy()
quarterly_common_size_labels = (
    quarterly_common_size_frame["date"].dt.year.astype(str)
    + " "
    + quarterly_common_size_frame["quarterLabel"].astype(str)
    + " ("
    + quarterly_common_size_frame["date"].dt.strftime("%b %Y")
    + ")"
    )
quarterly_common_size = build_common_size_statement(quarterly_common_size_frame, quarterly_common_size_labels)

display(
    annual_common_size.style.format("{:.1%}").set_caption(
        f"{SYMBOL} annual common-size income statement (% of revenue)"
    )
)
display(
    quarterly_common_size.style.format("{:.1%}").set_caption(
        f"{SYMBOL} quarterly common-size income statement (% of revenue)"
    )
)

In [ ]:
# 11. Analyze EPS and dilution
from IPython.display import display
from plotly.subplots import make_subplots

annual_eps_frame = annual_revenue.tail(min(len(annual_revenue), 8)).copy()
quarterly_eps_frame = quarterly_revenue.tail(min(len(quarterly_revenue), 12)).copy()
analysis_label = locals().get("chart_label", locals().get("company_name", SYMBOL))

def format_eps_summary_value(metric: str, value: float) -> str:
    if pd.isna(value):
        return "-"
    if "dilution" in metric.lower() or "growth" in metric.lower():
        return f"{value:.2%}"
    return f"{value:,.2f}"

eps_dilution_summary = pd.DataFrame(
    [
        {
            "metric": "Latest annual basic EPS",
            "value": annual_revenue.iloc[-1]["eps"],
            "asOf": annual_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest annual diluted EPS",
            "value": annual_revenue.iloc[-1]["epsDiluted"],
            "asOf": annual_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest annual dilution",
            "value": annual_revenue.iloc[-1]["dilutionPct"],
            "asOf": annual_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest quarterly basic EPS",
            "value": quarterly_revenue.iloc[-1]["eps"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest quarterly diluted EPS",
            "value": quarterly_revenue.iloc[-1]["epsDiluted"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Latest quarterly dilution",
            "value": quarterly_revenue.iloc[-1]["dilutionPct"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
        {
            "metric": "Quarterly diluted share YoY growth",
            "value": quarterly_revenue.iloc[-1]["dilutedSharesYoY"],
            "asOf": quarterly_revenue.iloc[-1]["date"].date(),
        },
    ]
)
eps_dilution_summary_display = eps_dilution_summary.copy()
eps_dilution_summary_display["value"] = [
    format_eps_summary_value(metric, value)
    for metric, value in zip(eps_dilution_summary_display["metric"], eps_dilution_summary_display["value"])
]
display(eps_dilution_summary_display)

eps_dilution_fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.08,
    horizontal_spacing=0.08,
    subplot_titles=(
        "Annual EPS",
        "Quarterly EPS",
        "Annual weighted-average shares",
        "Quarterly weighted-average shares",
        "Annual dilution",
        "Quarterly dilution",
    ),
)

eps_colors = {
    "basic": "#60a5fa",
    "diluted": "#f59e0b",
    "dilution": "#34d399",
    "share_growth": "#f87171",
}

for frame, col_number in [
    (annual_eps_frame, 1),
    (quarterly_eps_frame, 2),
]:
    eps_dilution_fig.add_trace(
        go.Scatter(
            x=frame["date"],
            y=frame["eps"],
            mode="lines+markers",
            name="Basic EPS",
            line={"color": eps_colors["basic"], "width": 2.5},
            marker={"size": 7},
            legendgroup="eps",
            showlegend=col_number == 1,
        ),
        row=1,
        col=col_number,
    )
    eps_dilution_fig.add_trace(
        go.Scatter(
            x=frame["date"],
            y=frame["epsDiluted"],
            mode="lines+markers",
            name="Diluted EPS",
            line={"color": eps_colors["diluted"], "width": 2.5, "dash": "dash"},
            marker={"size": 7},
            legendgroup="eps",
            showlegend=col_number == 1,
        ),
        row=1,
        col=col_number,
    )
    eps_dilution_fig.add_trace(
        go.Bar(
            x=frame["date"],
            y=frame["weightedAverageShsOut"] / 1_000_000_000,
            name="Basic shares",
            marker_color=eps_colors["basic"],
            opacity=0.75,
            legendgroup="shares",
            showlegend=col_number == 1,
        ),
        row=2,
        col=col_number,
    )
    eps_dilution_fig.add_trace(
        go.Bar(
            x=frame["date"],
            y=frame["weightedAverageShsOutDil"] / 1_000_000_000,
            name="Diluted shares",
            marker_color=eps_colors["diluted"],
            opacity=0.75,
            legendgroup="shares",
            showlegend=col_number == 1,
        ),
        row=2,
        col=col_number,
    )
    eps_dilution_fig.add_trace(
        go.Scatter(
            x=frame["date"],
            y=frame["dilutionPct"],
            mode="lines+markers",
            name="Dilution %",
            line={"color": eps_colors["dilution"], "width": 2.5},
            marker={"size": 7},
            legendgroup="dilution",
            showlegend=col_number == 1,
        ),
        row=3,
        col=col_number,
    )

apply_standard_figure_layout(eps_dilution_fig, f"{analysis_label} EPS and dilution", 1260)
eps_dilution_fig.update_layout(barmode="group")

for row_number in [1, 2, 3]:
    for col_number in [1, 2]:
        eps_dilution_fig.update_xaxes(
            showgrid=True,
            gridcolor="rgba(148, 163, 184, 0.18)",
            zeroline=False,
            automargin=True,
            row=row_number,
            col=col_number,
        )

eps_dilution_fig.update_yaxes(title_text="EPS", tickformat=",.2f", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=1, col=1)
eps_dilution_fig.update_yaxes(title_text="EPS", tickformat=",.2f", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=1, col=2)
eps_dilution_fig.update_yaxes(title_text="Shares (bn)", tickformat=",.1f", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=2, col=1)
eps_dilution_fig.update_yaxes(title_text="Shares (bn)", tickformat=",.1f", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=2, col=2)
eps_dilution_fig.update_yaxes(title_text="Dilution", tickformat=".2%", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=3, col=1)
eps_dilution_fig.update_yaxes(title_text="Dilution", tickformat=".2%", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=3, col=2)

eps_dilution_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 12. Track TTM profit conversion
from IPython.display import display
from plotly.subplots import make_subplots

ttm_frame = quarterly_revenue.dropna(subset=["ttmRevenue"]).copy()
analysis_label = locals().get("chart_label", locals().get("company_name", SYMBOL))

if ttm_frame.empty:
    raise RuntimeError(f"Need at least four quarters of history to compute TTM profit metrics for {SYMBOL}.")

def format_ttm_summary_value(metric: str, value: float) -> str:
    if pd.isna(value):
        return "-"
    if "margin" in metric.lower():
        return f"{value:.2%}"
    return f"{value:,.0f}"

latest_ttm_metrics = ttm_frame.iloc[-1]
ttm_summary = pd.DataFrame(
    [
        {
            "metric": "Latest TTM revenue",
            "value": latest_ttm_metrics["ttmRevenue"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM gross profit",
            "value": latest_ttm_metrics["ttmGrossProfit"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM operating income",
            "value": latest_ttm_metrics["ttmOperatingIncome"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM pre-tax income",
            "value": latest_ttm_metrics["ttmIncomeBeforeTax"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM net income",
            "value": latest_ttm_metrics["ttmNetIncome"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM gross margin",
            "value": latest_ttm_metrics["ttmGrossMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM operating margin",
            "value": latest_ttm_metrics["ttmOperatingMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM pre-tax margin",
            "value": latest_ttm_metrics["ttmIncomeBeforeTaxMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
        {
            "metric": "Latest TTM net margin",
            "value": latest_ttm_metrics["ttmNetMargin"],
            "asOf": latest_ttm_metrics["date"].date(),
        },
    ]
)
ttm_summary_display = ttm_summary.copy()
ttm_summary_display["value"] = [
    format_ttm_summary_value(metric, value)
    for metric, value in zip(ttm_summary_display["metric"], ttm_summary_display["value"])
]
display(ttm_summary_display)

ttm_profit_fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    row_heights=[0.36, 0.32, 0.32],
    subplot_titles=(
        "TTM profit dollars",
        "TTM profit margins",
        "TTM YoY growth",
    ),
)

profit_traces = [
    ("ttmGrossProfit", "Gross profit", "#60a5fa"),
    ("ttmOperatingIncome", "Operating income", "#34d399"),
    ("ttmIncomeBeforeTax", "Pre-tax income", "#f59e0b"),
    ("ttmNetIncome", "Net income", "#f87171"),
]
for metric_name, metric_label, color in profit_traces:
    ttm_profit_fig.add_trace(
        go.Scatter(
            x=ttm_frame["date"],
            y=ttm_frame[metric_name],
            mode="lines+markers",
            name=metric_label,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
            legendgroup="profit_dollars",
        ),
        row=1,
        col=1,
    )

margin_traces = [
    ("ttmGrossMargin", "Gross margin", "#60a5fa"),
    ("ttmOperatingMargin", "Operating margin", "#34d399"),
    ("ttmIncomeBeforeTaxMargin", "Pre-tax margin", "#f59e0b"),
    ("ttmNetMargin", "Net margin", "#f87171"),
]
for metric_name, metric_label, color in margin_traces:
    ttm_profit_fig.add_trace(
        go.Scatter(
            x=ttm_frame["date"],
            y=ttm_frame[metric_name],
            mode="lines+markers",
            name=metric_label,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
            legendgroup="profit_margin",
            showlegend=False,
        ),
        row=2,
        col=1,
    )

growth_traces = [
    ("ttmRevenueYoY", "Revenue growth", "#c084fc"),
    ("ttmGrossProfitYoY", "Gross profit growth", "#60a5fa"),
    ("ttmOperatingIncomeYoY", "Operating income growth", "#34d399"),
    ("ttmIncomeBeforeTaxYoY", "Pre-tax income growth", "#f59e0b"),
    ("ttmNetIncomeYoY", "Net income growth", "#f87171"),
]
for metric_name, metric_label, color in growth_traces:
    ttm_profit_fig.add_trace(
        go.Scatter(
            x=ttm_frame["date"],
            y=ttm_frame[metric_name],
            mode="lines+markers",
            name=metric_label,
            line={"color": color, "width": 2.5},
            marker={"size": 7},
            legendgroup="profit_growth",
            showlegend=False,
        ),
        row=3,
        col=1,
    )

apply_standard_figure_layout(ttm_profit_fig, f"{analysis_label} TTM profit conversion", 1220)

for row_number in [1, 2, 3]:
    ttm_profit_fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.18)",
        zeroline=False,
        automargin=True,
        row=row_number,
        col=1,
    )

ttm_profit_fig.update_yaxes(title_text="Amount", tickformat=",.2s", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=1, col=1)
ttm_profit_fig.update_yaxes(title_text="Margin", tickformat=".1%", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", row=2, col=1)
ttm_profit_fig.update_yaxes(title_text="YoY growth", tickformat=".1%", showgrid=True, gridcolor="rgba(148, 163, 184, 0.18)", zeroline=True, zerolinecolor="rgba(226, 232, 240, 0.35)", row=3, col=1)
ttm_profit_fig.update_xaxes(title_text="Quarter end", row=3, col=1)

ttm_profit_fig.show(config={"responsive": True, "displaylogo": False})

## Margin Driver Analysis

These cells decompose profitability into gross margin, operating margin, and net margin drivers. Use them to compare revenue growth against cost structure changes and to see how below-the-line items affect final net margins.

In [ ]:
# 10. Plot gross margin drivers
from plotly.subplots import make_subplots

gross_margin_fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
    row_heights=[0.38, 0.31, 0.31],
    subplot_titles=(
        "Annual Revenue vs Cost of revenue (COGS)",
        "Quarterly Revenue vs Cost of revenue (COGS)",
        "Annual Gross Margin and Cost of Revenue Ratio",
        "Quarterly Gross Margin and Cost of Revenue Ratio",
        "Annual Revenue Growth vs Cost of Revenue Growth",
        "Quarterly Revenue Growth vs Cost of Revenue Growth",
    ),
)

gross_margin_fig.add_trace(
    go.Bar(
        x=annual_revenue["date"],
        y=annual_revenue["revenue"],
        name="Annual revenue",
        marker_color="#60a5fa",
        opacity=0.55,
        legendgroup="annual-revenue",
    ),
    row=1,
    col=1,
)
gross_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["costOfRevenue"],
        name="Annual cost of revenue (COGS)",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2.5},
        marker={"size": 7},
        legendgroup="annual-cost-of-revenue",
    ),
    row=1,
    col=1,
)

gross_margin_fig.add_trace(
    go.Bar(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["revenue"],
        name="Quarterly revenue",
        marker_color="#38bdf8",
        opacity=0.5,
        legendgroup="quarterly-revenue",
    ),
    row=1,
    col=2,
)
gross_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["costOfRevenue"],
        name="Quarterly cost of revenue (COGS)",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.25},
        marker={"size": 6},
        legendgroup="quarterly-cost-of-revenue",
    ),
    row=1,
    col=2,
)

gross_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["grossMargin"],
        name="Annual gross margin",
        mode="lines+markers",
        line={"color": "#a78bfa", "width": 2.5},
        marker={"size": 7},
        legendgroup="gross-margin",
    ),
    row=2,
    col=1,
)
gross_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["costOfRevenuePctRevenue"],
        name="Annual cost of revenue % revenue",
        mode="lines+markers",
        line={"color": "#f43f5e", "width": 2.25, "dash": "dash"},
        marker={"size": 6},
        legendgroup="cost-of-revenue-ratio",
    ),
    row=2,
    col=1,
)

gross_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["grossMargin"],
        name="Quarterly gross margin",
        mode="lines+markers",
        line={"color": "#8b5cf6", "width": 2.5},
        marker={"size": 6},
        legendgroup="gross-margin",
    ),
    row=2,
    col=2,
)
gross_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["costOfRevenuePctRevenue"],
        name="Quarterly cost of revenue % revenue",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2.25, "dash": "dash"},
        marker={"size": 6},
        legendgroup="cost-of-revenue-ratio",
    ),
    row=2,
    col=2,
)

gross_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["revenueYoY"],
        name="Annual revenue growth",
        mode="lines+markers",
        line={"color": "#38bdf8", "width": 2.5},
        marker={"size": 7},
        legendgroup="growth",
    ),
    row=3,
    col=1,
)
gross_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["costOfRevenueYoY"],
        name="Annual cost of revenue growth",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2.25, "dash": "dash"},
        marker={"size": 6},
        legendgroup="growth",
    ),
    row=3,
    col=1,
)

gross_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["revenueYoY"],
        name="Quarterly revenue growth",
        mode="lines+markers",
        line={"color": "#38bdf8", "width": 2.5},
        marker={"size": 6},
        legendgroup="growth",
    ),
    row=3,
    col=2,
)
gross_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["costOfRevenueYoY"],
        name="Quarterly cost of revenue growth",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.25, "dash": "dash"},
        marker={"size": 6},
        legendgroup="growth",
    ),
    row=3,
    col=2,
)

apply_standard_figure_layout(gross_margin_fig, f"{chart_label} gross margin drivers", 1320)

for row_number in [1, 2, 3]:
    for col_number in [1, 2]:
        gross_margin_fig.update_xaxes(
            showgrid=True,
            gridcolor="rgba(148, 163, 184, 0.18)",
            zeroline=False,
            automargin=True,
            row=row_number,
            col=col_number,
        )

gross_margin_fig.update_yaxes(
    title_text="USD",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
    tickformat="$,.3s",
    row=1,
    col=1,
)
gross_margin_fig.update_yaxes(
    title_text="USD",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
    tickformat="$,.3s",
    row=1,
    col=2,
)
gross_margin_fig.update_yaxes(
    title_text="Percent of revenue",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=2,
    col=1,
)
gross_margin_fig.update_yaxes(
    title_text="Percent of revenue",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=2,
    col=2,
)
gross_margin_fig.update_yaxes(
    title_text="Growth",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=3,
    col=1,
)
gross_margin_fig.update_yaxes(
    title_text="Growth",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=3,
    col=2,
)

gross_margin_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 11. Plot operating margin drivers
from plotly.subplots import make_subplots

operating_margin_fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
    row_heights=[0.38, 0.31, 0.31],
    subplot_titles=(
        "Annual Gross Profit vs Operating Expenses",
        "Quarterly Gross Profit vs Operating Expenses",
        "Annual Operating Margin Drivers",
        "Quarterly Operating Margin Drivers",
        "Annual Revenue Growth vs Operating Expense Growth",
        "Quarterly Revenue Growth vs Operating Expense Growth",
    ),
)

operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["grossProfit"],
        name="Annual gross profit",
        mode="lines+markers",
        line={"color": "#a78bfa", "width": 2.5},
        marker={"size": 7},
        legendgroup="gross-profit",
    ),
    row=1,
    col=1,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingExpenses"],
        name="Annual operating expenses",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2.5},
        marker={"size": 7},
        legendgroup="opex",
    ),
    row=1,
    col=1,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingIncome"],
        name="Annual operating income",
        mode="lines+markers",
        line={"color": "#22c55e", "width": 2.5},
        marker={"size": 7},
        legendgroup="op-income",
    ),
    row=1,
    col=1,
)

operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["grossProfit"],
        name="Quarterly gross profit",
        mode="lines+markers",
        line={"color": "#c084fc", "width": 2.25},
        marker={"size": 6},
        legendgroup="gross-profit",
    ),
    row=1,
    col=2,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingExpenses"],
        name="Quarterly operating expenses",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.25},
        marker={"size": 6},
        legendgroup="opex",
    ),
    row=1,
    col=2,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingIncome"],
        name="Quarterly operating income",
        mode="lines+markers",
        line={"color": "#4ade80", "width": 2.25},
        marker={"size": 6},
        legendgroup="op-income",
    ),
    row=1,
    col=2,
)

operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingMargin"],
        name="Annual operating margin",
        mode="lines+markers",
        line={"color": "#38bdf8", "width": 2.5},
        marker={"size": 7},
        legendgroup="op-margin",
    ),
    row=2,
    col=1,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingExpensesPctRevenue"],
        name="Annual operating expenses % revenue",
        mode="lines+markers",
        line={"color": "#f97316", "width": 2.25, "dash": "dash"},
        marker={"size": 6},
        legendgroup="opex-ratio",
    ),
    row=2,
    col=1,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["sgaPctRevenue"],
        name="Annual SG&A % revenue",
        mode="lines+markers",
        line={"color": "#f59e0b", "width": 2},
        marker={"size": 6},
        legendgroup="sga-ratio",
    ),
    row=2,
    col=1,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["rdPctRevenue"],
        name="Annual R&D % revenue",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2},
        marker={"size": 6},
        legendgroup="rd-ratio",
    ),
    row=2,
    col=1,
)

operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingMargin"],
        name="Quarterly operating margin",
        mode="lines+markers",
        line={"color": "#0ea5e9", "width": 2.25},
        marker={"size": 6},
        legendgroup="op-margin",
    ),
    row=2,
    col=2,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingExpensesPctRevenue"],
        name="Quarterly operating expenses % revenue",
        mode="lines+markers",
        line={"color": "#fb923c", "width": 2.25, "dash": "dash"},
        marker={"size": 6},
        legendgroup="opex-ratio",
    ),
    row=2,
    col=2,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["sgaPctRevenue"],
        name="Quarterly SG&A % revenue",
        mode="lines+markers",
        line={"color": "#fbbf24", "width": 2},
        marker={"size": 6},
        legendgroup="sga-ratio",
    ),
    row=2,
    col=2,
)
operating_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["rdPctRevenue"],
        name="Quarterly R&D % revenue",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2},
        marker={"size": 6},
        legendgroup="rd-ratio",
    ),
    row=2,
    col=2,
)

operating_growth_traces = [
    (
        annual_revenue,
        "date",
        [
            ("revenueYoY", "Annual revenue growth", "#38bdf8", 2.5, None),
            ("operatingExpensesYoY", "Annual operating expenses growth", "#f97316", 2.25, "dash"),
            ("sgaYoY", "Annual SG&A growth", "#f59e0b", 2, None),
            ("rdYoY", "Annual R&D growth", "#fb7185", 2, None),
        ],
        3,
        1,
    ),
    (
        quarterly_revenue,
        "date",
        [
            ("revenueYoY", "Quarterly revenue growth", "#38bdf8", 2.5, None),
            ("operatingExpensesYoY", "Quarterly operating expenses growth", "#fb923c", 2.25, "dash"),
            ("sgaYoY", "Quarterly SG&A growth", "#fbbf24", 2, None),
            ("rdYoY", "Quarterly R&D growth", "#fb7185", 2, None),
        ],
        3,
        2,
    ),
]

for frame, x_column, trace_specs, row_number, col_number in operating_growth_traces:
    for metric_name, label, color, width, dash in trace_specs:
        metric_series = frame[metric_name]
        if metric_series.notna().any():
            line_style = {"color": color, "width": width}
            if dash:
                line_style["dash"] = dash

            operating_margin_fig.add_trace(
                go.Scatter(
                    x=frame[x_column],
                    y=metric_series,
                    name=label,
                    mode="lines+markers",
                    line=line_style,
                    marker={"size": 6},
                    legendgroup="growth",
                ),
                row=row_number,
                col=col_number,
            )

apply_standard_figure_layout(operating_margin_fig, f"{chart_label} operating margin drivers", 1320)

for row_number in [1, 2, 3]:
    for col_number in [1, 2]:
        operating_margin_fig.update_xaxes(
            showgrid=True,
            gridcolor="rgba(148, 163, 184, 0.18)",
            zeroline=False,
            automargin=True,
            row=row_number,
            col=col_number,
        )

operating_margin_fig.update_yaxes(
    title_text="USD",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
    tickformat="$,.3s",
    row=1,
    col=1,
)
operating_margin_fig.update_yaxes(
    title_text="USD",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
    tickformat="$,.3s",
    row=1,
    col=2,
)
operating_margin_fig.update_yaxes(
    title_text="Percent of revenue",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=2,
    col=1,
)
operating_margin_fig.update_yaxes(
    title_text="Percent of revenue",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=2,
    col=2,
)
operating_margin_fig.update_yaxes(
    title_text="Growth",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=3,
    col=1,
)
operating_margin_fig.update_yaxes(
    title_text="Growth",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=3,
    col=2,
)

operating_margin_fig.show(config={"responsive": True, "displaylogo": False})

In [ ]:
# 12. Plot net margin drivers
from plotly.subplots import make_subplots

net_margin_fig = make_subplots(
    rows=3,
    cols=2,
    vertical_spacing=0.1,
    horizontal_spacing=0.08,
    row_heights=[0.38, 0.31, 0.31],
    subplot_titles=(
        "Annual Operating Income vs Pretax Income vs Net Income",
        "Quarterly Operating Income vs Pretax Income vs Net Income",
        "Annual Net Margin and Below-the-line Ratios",
        "Quarterly Net Margin and Below-the-line Ratios",
        "Annual Operating Income vs Pretax vs Net Income Growth",
        "Quarterly Operating Income vs Pretax vs Net Income Growth",
    ),
)

net_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["operatingIncome"],
        name="Annual operating income",
        mode="lines+markers",
        line={"color": "#22c55e", "width": 2.5},
        marker={"size": 7},
        legendgroup="income-levels",
    ),
    row=1,
    col=1,
)
net_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["incomeBeforeTax"],
        name="Annual pretax income",
        mode="lines+markers",
        line={"color": "#38bdf8", "width": 2.5},
        marker={"size": 7},
        legendgroup="income-levels",
    ),
    row=1,
    col=1,
)
net_margin_fig.add_trace(
    go.Scatter(
        x=annual_revenue["date"],
        y=annual_revenue["netIncome"],
        name="Annual net income",
        mode="lines+markers",
        line={"color": "#f43f5e", "width": 2.5},
        marker={"size": 7},
        legendgroup="income-levels",
    ),
    row=1,
    col=1,
)

net_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["operatingIncome"],
        name="Quarterly operating income",
        mode="lines+markers",
        line={"color": "#4ade80", "width": 2.25},
        marker={"size": 6},
        legendgroup="income-levels",
    ),
    row=1,
    col=2,
)
net_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["incomeBeforeTax"],
        name="Quarterly pretax income",
        mode="lines+markers",
        line={"color": "#0ea5e9", "width": 2.25},
        marker={"size": 6},
        legendgroup="income-levels",
    ),
    row=1,
    col=2,
)
net_margin_fig.add_trace(
    go.Scatter(
        x=quarterly_revenue["date"],
        y=quarterly_revenue["netIncome"],
        name="Quarterly net income",
        mode="lines+markers",
        line={"color": "#fb7185", "width": 2.25},
        marker={"size": 6},
        legendgroup="income-levels",
    ),
    row=1,
    col=2,
)

net_margin_ratio_traces = [
    (
        annual_revenue,
        "date",
        [
            ("operatingMargin", "Annual operating margin", "#22c55e", 2.5, None, "margin-ratios"),
            ("incomeBeforeTaxMargin", "Annual pretax margin", "#38bdf8", 2.25, None, "margin-ratios"),
            ("netMargin", "Annual net margin", "#f43f5e", 2.5, None, "margin-ratios"),
            ("otherIncomeExpensePctRevenue", "Annual other income / expense % revenue", "#a78bfa", 2, "dash", "below-line-ratios"),
            ("incomeTaxExpensePctRevenue", "Annual tax expense % revenue", "#f97316", 2, "dash", "below-line-ratios"),
        ],
        2,
        1,
    ),
    (
        quarterly_revenue,
        "date",
        [
            ("operatingMargin", "Quarterly operating margin", "#4ade80", 2.25, None, "margin-ratios"),
            ("incomeBeforeTaxMargin", "Quarterly pretax margin", "#0ea5e9", 2.25, None, "margin-ratios"),
            ("netMargin", "Quarterly net margin", "#fb7185", 2.25, None, "margin-ratios"),
            ("otherIncomeExpensePctRevenue", "Quarterly other income / expense % revenue", "#c084fc", 2, "dash", "below-line-ratios"),
            ("incomeTaxExpensePctRevenue", "Quarterly tax expense % revenue", "#fb923c", 2, "dash", "below-line-ratios"),
        ],
        2,
        2,
    ),
]

for frame, x_column, trace_specs, row_number, col_number in net_margin_ratio_traces:
    for metric_name, label, color, width, dash, legend_group in trace_specs:
        metric_series = frame[metric_name]
        if metric_series.notna().any():
            line_style = {"color": color, "width": width}
            if dash:
                line_style["dash"] = dash

            net_margin_fig.add_trace(
                go.Scatter(
                    x=frame[x_column],
                    y=metric_series,
                    name=label,
                    mode="lines+markers",
                    line=line_style,
                    marker={"size": 6},
                    legendgroup=legend_group,
                ),
                row=row_number,
                col=col_number,
            )

net_margin_growth_traces = [
    (
        annual_revenue,
        "date",
        [
            ("operatingIncomeYoY", "Annual operating income growth", "#22c55e", 2.5, None),
            ("incomeBeforeTaxYoY", "Annual pretax income growth", "#38bdf8", 2.25, None),
            ("netIncomeYoY", "Annual net income growth", "#f43f5e", 2.5, None),
            ("incomeTaxExpenseYoY", "Annual tax expense growth", "#f97316", 2, "dash"),
        ],
        3,
        1,
    ),
    (
        quarterly_revenue,
        "date",
        [
            ("operatingIncomeYoY", "Quarterly operating income growth", "#4ade80", 2.25, None),
            ("incomeBeforeTaxYoY", "Quarterly pretax income growth", "#0ea5e9", 2.25, None),
            ("netIncomeYoY", "Quarterly net income growth", "#fb7185", 2.25, None),
            ("incomeTaxExpenseYoY", "Quarterly tax expense growth", "#fb923c", 2, "dash"),
        ],
        3,
        2,
    ),
]

for frame, x_column, trace_specs, row_number, col_number in net_margin_growth_traces:
    for metric_name, label, color, width, dash in trace_specs:
        metric_series = frame[metric_name]
        if metric_series.notna().any():
            line_style = {"color": color, "width": width}
            if dash:
                line_style["dash"] = dash

            net_margin_fig.add_trace(
                go.Scatter(
                    x=frame[x_column],
                    y=metric_series,
                    name=label,
                    mode="lines+markers",
                    line=line_style,
                    marker={"size": 6},
                    legendgroup="growth",
                ),
                row=row_number,
                col=col_number,
            )

apply_standard_figure_layout(net_margin_fig, f"{chart_label} net margin drivers", 1320)

for row_number in [1, 2, 3]:
    for col_number in [1, 2]:
        net_margin_fig.update_xaxes(
            showgrid=True,
            gridcolor="rgba(148, 163, 184, 0.18)",
            zeroline=False,
            automargin=True,
            row=row_number,
            col=col_number,
        )

net_margin_fig.update_yaxes(
    title_text="USD",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
    tickformat="$,.3s",
    row=1,
    col=1,
)
net_margin_fig.update_yaxes(
    title_text="USD",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=False,
    automargin=True,
    tickformat="$,.3s",
    row=1,
    col=2,
)
net_margin_fig.update_yaxes(
    title_text="Percent of revenue",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=2,
    col=1,
)
net_margin_fig.update_yaxes(
    title_text="Percent of revenue",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=2,
    col=2,
)
net_margin_fig.update_yaxes(
    title_text="Growth",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=3,
    col=1,
)
net_margin_fig.update_yaxes(
    title_text="Growth",
    showgrid=True,
    gridcolor="rgba(148, 163, 184, 0.18)",
    zeroline=True,
    zerolinecolor="rgba(226, 232, 240, 0.35)",
    automargin=True,
    tickformat=".1%",
    row=3,
    col=2,
)

net_margin_fig.show(config={"responsive": True, "displaylogo": False})

## Revenue Segmentation

This section loads any non-plan-gated annual revenue segmentation returned by FMP and visualizes the available product and geographic breakdowns. If segmentation data is unavailable for the current ticker or plan, the downstream cells will show that through the loaded availability checks.

In [ ]:
# 13. Load annual revenue segmentation

def load_revenue_segmentation_frame(endpoint: str) -> tuple[pd.DataFrame, list[str]]:
    try:
        payload = request_json(endpoint, symbol=SYMBOL, period="annual", structure="flat")
    except Exception:
        return pd.DataFrame(), []

    if not isinstance(payload, list) or not payload:
        return pd.DataFrame(), []

    records = []
    for item in payload:
        data = item.get("data") or {}
        if not data:
            continue

        row = {
            "date": pd.to_datetime(item.get("date"), errors="coerce"),
            "fiscalYear": pd.to_numeric(item.get("fiscalYear"), errors="coerce"),
            "period": item.get("period"),
        }
        row.update({key: pd.to_numeric(value, errors="coerce") for key, value in data.items()})
        records.append(row)

    frame = pd.DataFrame(records)
    if frame.empty:
        return frame, []

    frame = frame.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    segment_columns = [column for column in frame.columns if column not in {"date", "fiscalYear", "period"}]
    if not segment_columns:
        return pd.DataFrame(), []

    segment_columns = (
        frame.loc[:, segment_columns]
        .fillna(0)
        .iloc[-1]
        .sort_values(ascending=False)
        .index
        .tolist()
    )
    frame["totalSegmentRevenue"] = frame[segment_columns].sum(axis=1, min_count=1)
    return frame, segment_columns


annual_product_segmentation, annual_product_segments = load_revenue_segmentation_frame("revenue-product-segmentation")
annual_geographic_segmentation, annual_geographic_segments = load_revenue_segmentation_frame("revenue-geographic-segmentation")

available_revenue_segmentations = pd.Series(
    {
        "annualProduct": bool(annual_product_segments),
        "annualGeographic": bool(annual_geographic_segments),
    },
    name="available",
)
available_revenue_segmentations

In [ ]:
# 14. Plot annual revenue segmentation
from plotly.subplots import make_subplots

segmentation_specs = []
if annual_product_segments:
    segmentation_specs.append(
        (
            annual_product_segmentation,
            annual_product_segments,
            "Annual revenue by product",
            ["#60a5fa", "#38bdf8", "#34d399", "#f59e0b", "#f97316", "#f43f5e", "#a78bfa", "#818cf8"],
        )
    )
if annual_geographic_segments:
    segmentation_specs.append(
        (
            annual_geographic_segmentation,
            annual_geographic_segments,
            "Annual revenue by geography",
            ["#22c55e", "#14b8a6", "#0ea5e9", "#8b5cf6", "#ec4899", "#f97316", "#eab308", "#fb7185"],
        )
    )

if not segmentation_specs:
    raise RuntimeError(f"FMP did not return annual revenue segmentation for {SYMBOL}.")

revenue_segmentation_fig = make_subplots(
    rows=len(segmentation_specs),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.12 if len(segmentation_specs) > 1 else 0.08,
    subplot_titles=tuple(spec[2] for spec in segmentation_specs),
)

for row_number, (segment_frame, segment_columns, subplot_title, palette) in enumerate(segmentation_specs, start=1):
    for color_index, segment_name in enumerate(segment_columns):
        revenue_segmentation_fig.add_trace(
            go.Bar(
                x=segment_frame["date"],
                y=segment_frame[segment_name],
                name=segment_name,
                marker_color=palette[color_index % len(palette)],
                legendgroup=subplot_title,
            ),
            row=row_number,
            col=1,
        )

    revenue_segmentation_fig.add_trace(
        go.Scatter(
            x=segment_frame["date"],
            y=segment_frame["totalSegmentRevenue"],
            name=f"{subplot_title} total",
            mode="lines+markers",
            line={"color": "#e2e8f0", "width": 2.5},
            marker={"size": 7},
            legendgroup=f"{subplot_title}-total",
        ),
        row=row_number,
        col=1,
    )

figure_height = 620 if len(segmentation_specs) == 1 else 1120
apply_standard_figure_layout(revenue_segmentation_fig, f"{chart_label} annual revenue segmentation", figure_height, bottom_margin=90)
revenue_segmentation_fig.update_layout(
    barmode="stack",
    legend={
        "orientation": "v",
        "yanchor": "top",
        "y": 1,
        "xanchor": "left",
        "x": 1.02,
        "bgcolor": "rgba(2, 8, 23, 0.6)",
    },
    margin={"l": 60, "r": 240, "t": 120, "b": 80},
)

for row_number in range(1, len(segmentation_specs) + 1):
    revenue_segmentation_fig.update_xaxes(
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.18)",
        zeroline=False,
        automargin=True,
        row=row_number,
        col=1,
    )
    revenue_segmentation_fig.update_yaxes(
        title_text="Revenue (USD)",
        showgrid=True,
        gridcolor="rgba(148, 163, 184, 0.18)",
        zeroline=False,
        automargin=True,
        tickformat="$,.3s",
        row=row_number,
        col=1,
    )

revenue_segmentation_fig.update_xaxes(title_text="Date", row=len(segmentation_specs), col=1)

revenue_segmentation_fig.show(config={"responsive": True, "displaylogo": False})